In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from bound_propagation import BoundModelFactory, HyperRectangle
from tqdm import tqdm

In [3]:
# !pip3 install bound-propagation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available()  else "cpu")
batch_size = 64

In [5]:
np.random.seed(42)
torch.manual_seed(42)

In [6]:
## Dataloaders - No normalization for IBP
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True,
                              transform=transforms.Compose([transforms.ToTensor()]))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True,
                             transform=transforms.Compose([transforms.ToTensor()]))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [7]:
class Net(nn.Module):
  def __init__(self, hidden_size=50):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(28*28, hidden_size)
    self.fc2 = nn.Linear(hidden_size, hidden_size)
    self.fc3 = nn.Linear(hidden_size, hidden_size)
    self.fc4 = nn.Linear(hidden_size, 10) # Output layer

  def forward(self, x):
    x = x.view((-1, 28*28))
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = self.fc4(x)
    return x

In [8]:
model = Net().to(device)
model.train()

Net(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (fc4): Linear(in_features=50, out_features=10, bias=True)
)

In [9]:
def compute_logits_worst_case(model, x, y, eps):
    n = x.size(0)
    flat_x = x.reshape(n, -1)

    # Define perturbed input range [x - eps, x + eps], clamped to valid pixel range
    lower = torch.clamp(flat_x - eps, 0.0, 1.0)
    upper = torch.clamp(flat_x + eps, 0.0, 1.0)
    bounds = HyperRectangle(lower, upper)

    # Reconstruct the sequential feedforward network
    layers = [
        model.fc1, nn.ReLU(),
        model.fc2, nn.ReLU(),
        model.fc3, nn.ReLU(),
        model.fc4
    ]
    sequential_net = nn.Sequential(*layers)

    # Initialize bounded model via factory
    bounded_model = BoundModelFactory().build(sequential_net)

    # Propagate bounds to compute logits interval
    logits_interval = bounded_model.ibp(bounds)
    lo, hi = logits_interval.lower, logits_interval.upper

    # Select lower bounds for true class, upper for all others
    idx = torch.arange(n)
    selector = torch.ones_like(lo, dtype=torch.bool)
    selector[idx, y] = False
    combined_logits = torch.where(selector, hi, lo)

    # Compute cross-entropy over worst-case logits
    loss = F.cross_entropy(combined_logits, y)
    return loss

### Part A

In [14]:
def train_ibp(model, train_loader, epochs=20, max_eps=0.1, device='cuda'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    log = {'loss': [], 'std_loss': [], 'rob_loss': []}

    print(f"Starting IBP training for {epochs} epochs (target ε={max_eps})")

    for ep in range(epochs):
        model.train()
        total_loss = total_std = total_rob = 0.0

        # Linear schedules
        kappa = 1.0 - 0.5 * (ep / epochs)
        eps = max_eps * (ep / epochs)

        for images, labels in tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            # Standard CE
            logits = model(images)
            loss_std = F.cross_entropy(logits, labels)

            # Robust CE via IBP
            loss_rob = compute_logits_worst_case(model, images, labels, eps)

            # Combined objective
            loss = kappa * loss_std + (1 - kappa) * loss_rob
            loss.backward()
            optimizer.step()

        model.eval()
        tot_val, tot_acc = 0.0, 0.0
        val_loss = 0.0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            with torch.no_grad():
                logits = model(images)
                ce_loss = nn.CrossEntropyLoss()(logits, labels)
                rb_loss = compute_logits_worst_case(model, images, labels, eps)
                loss = kappa * ce_loss + (1 - kappa) * rb_loss
                val_loss += loss.item()
                tot_acc += (logits.argmax(dim=1) == labels).sum().item()
                tot_val += labels.size(0)
        val_acc = 100.0 * tot_acc / tot_val

        print(f'Epoch {ep+1}/{epochs}, Loss: {val_loss/len(train_loader):.3f}, Accuracy: {val_acc:.2f}%')

        # print(f"Epoch {ep+1:2d}/{epochs} → Loss={avg_loss:.4f}, Std={avg_std:.4f}, Rob={avg_rob:.4f}")

    return log


# === Example usage ===
start = time.time()
history = train_ibp(model, train_loader, epochs=20, max_eps=0.1, device=device)
ibp_tt = time.time() - start

print(f"\nIBP Training completed in {ibp_tt:.2f}s ({ibp_tt/60:.2f}m)")

Starting IBP training for 20 epochs (target ε=0.1)


Epoch 1/20: 100%|██████████| 938/938 [00:03<00:00, 249.56it/s]


Epoch 1/20, Loss: 0.092, Accuracy: 97.17%


Epoch 2/20:  37%|███▋      | 349/938 [00:01<00:02, 236.45it/s]


KeyboardInterrupt: 

In [ ]:
def train_standard(model, train_loader, epochs=5, lr=1e-3, device='cuda'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    start = time.time()

    for ep in range(epochs):
        model.train()
        total_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            print(model(images))
            print(labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f'Epoch {ep+1}/{epochs}, Loss: {total_loss/len(train_loader):.3f}')

    return time.time() - start


# === Example usage and timing comparison ===
std_model = Net().to(device)
standard_tt = train_standard(std_model, train_loader, epochs=20, lr=1e-3, device=device)

diff = ibp_tt - standard_tt
print(f"\nTraining Time Comparison:")
print(f"  IBP: {ibp_tt:.2f}s ({ibp_tt/60:.2f}m)")
print(f"  Standard: {standard_tt:.2f}s ({standard_tt/60:.2f}m)")


Epoch 1/20:   0%|          | 0/938 [00:00<?, ?it/s]


tensor([[ 1.8727e-01, -1.3209e-02,  7.4516e-02,  6.8734e-02, -4.6380e-02,
         -8.8713e-03,  3.1458e-02,  1.1947e-01, -2.2609e-02,  1.0335e-01],
        [ 1.7813e-01, -7.9515e-03,  7.8004e-02,  6.2014e-02, -4.9423e-02,
         -8.9813e-03,  2.4728e-02,  1.1157e-01, -2.9523e-02,  1.0001e-01],
        [ 1.9976e-01, -2.0156e-02,  8.3598e-02,  6.4407e-02, -4.2874e-02,
         -1.3522e-02,  2.8986e-02,  1.1516e-01, -2.9351e-02,  1.0860e-01],
        [ 1.9783e-01, -2.3009e-02,  7.9572e-02,  6.4112e-02, -3.7375e-02,
         -1.0857e-02,  1.6679e-02,  1.0455e-01, -2.4943e-02,  1.0100e-01],
        [ 1.7530e-01, -1.0726e-02,  7.3831e-02,  6.3816e-02, -5.1005e-02,
         -1.0634e-02,  2.2616e-02,  1.1223e-01, -2.9413e-02,  9.8754e-02],
        [ 1.9836e-01, -1.7615e-02,  7.1975e-02,  6.9193e-02, -5.4555e-02,
         -1.7205e-02,  3.5414e-02,  1.2442e-01, -1.9959e-02,  1.0028e-01],
        [ 1.8507e-01, -1.5822e-02,  7.5163e-02,  7.4274e-02, -3.8143e-02,
         -1.3277e-02,  2.5809e-0

Epoch 2/20:   0%|          | 0/938 [00:00<?, ?it/s]


tensor([[ 1.8003e-01, -1.6454e-02,  7.7410e-02,  6.8907e-02, -4.5112e-02,
         -7.6172e-03,  2.1761e-02,  1.1471e-01, -3.4400e-02,  1.0899e-01],
        [ 1.8128e-01, -1.7156e-02,  8.5985e-02,  5.3305e-02, -4.3930e-02,
         -8.1810e-03,  5.1797e-03,  9.5363e-02, -2.9028e-02,  1.0637e-01],
        [ 1.8603e-01, -1.2300e-02,  8.1492e-02,  7.3341e-02, -4.2365e-02,
         -1.6173e-02,  3.5726e-02,  1.2062e-01, -3.0421e-02,  9.8150e-02],
        [ 1.7920e-01, -3.4879e-03,  7.8177e-02,  5.4892e-02, -4.7068e-02,
         -8.6806e-03,  1.6145e-02,  1.1378e-01, -2.4439e-02,  1.0053e-01],
        [ 1.8783e-01, -6.7390e-03,  8.0098e-02,  6.2258e-02, -4.7568e-02,
         -1.8790e-02,  2.6519e-02,  1.1195e-01, -3.0692e-02,  9.4290e-02],
        [ 1.7642e-01, -7.1239e-03,  8.2129e-02,  6.4243e-02, -4.4786e-02,
         -2.0632e-03,  2.0326e-02,  1.1555e-01, -4.2912e-02,  1.0034e-01],
        [ 1.8184e-01, -7.3846e-03,  7.9339e-02,  6.5553e-02, -4.1459e-02,
         -1.0178e-02,  1.8438e-0

Epoch 3/20:   0%|          | 0/938 [00:00<?, ?it/s]


...

Epoch 19/20:   0%|          | 0/938 [00:00<?, ?it/s]


tensor([[ 0.1973, -0.0027,  0.0873,  0.0500, -0.0400, -0.0152,  0.0087,  0.1037,
         -0.0277,  0.0955],
        [ 0.1765, -0.0211,  0.0935,  0.0741, -0.0277, -0.0134,  0.0365,  0.1075,
         -0.0480,  0.1030],
        [ 0.1964, -0.0159,  0.0857,  0.0633, -0.0413, -0.0087,  0.0177,  0.1089,
         -0.0229,  0.1015],
        [ 0.2004, -0.0120,  0.0763,  0.0676, -0.0430, -0.0121,  0.0209,  0.1059,
         -0.0285,  0.0955],
        [ 0.1851, -0.0081,  0.0740,  0.0675, -0.0428, -0.0102,  0.0205,  0.1145,
         -0.0237,  0.0988],
        [ 0.1821, -0.0089,  0.0841,  0.0707, -0.0460, -0.0084,  0.0225,  0.1205,
         -0.0410,  0.1035],
        [ 0.1933, -0.0185,  0.0752,  0.0552, -0.0492, -0.0141,  0.0207,  0.1060,
         -0.0182,  0.1018],
        [ 0.1873, -0.0148,  0.0793,  0.0666, -0.0468, -0.0006,  0.0205,  0.1099,
         -0.0367,  0.0956],
        [ 0.1755, -0.0091,  0.0842,  0.0651, -0.0461, -0.0081,  0.0253,  0.1162,
         -0.0394,  0.1069],
        [ 0.1962, -

Epoch 20/20:   0%|          | 0/938 [00:00<?, ?it/s]

tensor([[ 1.9875e-01, -5.3896e-03,  7.9487e-02,  6.7450e-02, -4.5717e-02,
         -2.2313e-02,  2.8945e-02,  1.1908e-01, -2.5261e-02,  9.2952e-02],
        [ 1.8141e-01, -1.2143e-02,  8.4796e-02,  7.3964e-02, -4.0838e-02,
         -7.7666e-03,  3.1273e-02,  1.1627e-01, -3.9546e-02,  1.0001e-01],
        [ 2.0997e-01, -1.7757e-02,  8.0429e-02,  6.8739e-02, -3.7089e-02,
         -2.2383e-02,  3.5236e-02,  1.1624e-01, -1.8644e-02,  8.4766e-02],
        [ 1.9524e-01, -1.0439e-02,  7.6063e-02,  6.9209e-02, -4.7070e-02,
         -1.4880e-02,  3.2249e-02,  1.1576e-01, -3.2883e-02,  8.9298e-02],
        [ 1.7742e-01, -5.6965e-03,  8.3281e-02,  6.2203e-02, -5.5449e-02,
          7.8577e-03,  1.3789e-02,  1.2179e-01, -3.6246e-02,  1.0437e-01],
        [ 1.8249e-01, -5.3086e-03,  7.4677e-02,  7.4455e-02, -5.1209e-02,
          1.7307e-04,  2.4020e-02,  1.2534e-01, -4.2653e-02,  9.6245e-02],
        [ 1.9175e-01, -1.1841e-02,  7.5908e-02,  6.8136e-02, -4.7128e-02,
         -1.5551e-02,  2.0826e-0

NameError: name 'ibp_tt' is not defined

In [10]:
def pgd_attack(model, x, y, eps: float = 0.1, step_size: float = 0.01, steps: int = 40):
    model.eval()
    batch_size = x.size(0)

    # Flatten input for consistent gradient operations
    x_orig = x.view(batch_size, -1)
    x_adv = x_orig.clone().detach()

    for _ in range(steps):
        x_adv.requires_grad_(True)

        logits = model(x_adv.view_as(x))
        loss = F.cross_entropy(logits, y)

        # Compute gradients w.r.t input
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + step_size * grad.sign()

        # Project perturbation back to ε-ball
        delta = torch.clamp(x_adv - x_orig, min=-eps, max=eps)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0)

    # Reshape back to original input dimensions
    return x_adv.view_as(x)


In [13]:
def evaluate_accuracy_and_robustness(model, loader, eps=0.1, device="cuda"):
    model.eval()
    correct_clean = correct_adv = total = 0

    for x, y in tqdm(loader, desc="Evaluating"):
        x, y = x.to(device), y.to(device)
        total += y.size(0)

        with torch.no_grad():
            clean_pred = model(x).argmax(1)
            correct_clean += (clean_pred == y).sum().item()

        x_adv = pgd_attack(model, x, y, eps=eps, step_size=0.01, steps=40)
        with torch.no_grad():
            adv_pred = model(x_adv).argmax(1)
            correct_adv += (adv_pred == y).sum().item()

    std_acc = 100 * correct_clean / total
    rob_acc = 100 * correct_adv / total
    return std_acc, rob_acc


# === Evaluate both models ===
ibp_std_acc, ibp_rob_acc = evaluate_accuracy_and_robustness(model, test_loader, eps=0.1, device=device)
std_std_acc, std_rob_acc = evaluate_accuracy_and_robustness(std_model, test_loader, eps=0.1, device=device)

print("\nResults Summary")
print("-" * 50)
print(f"IBP Model     - Standard: {ibp_std_acc:.2f}% | Robust: {ibp_rob_acc:.2f}%")
print(f"Standard Model- Standard: {std_std_acc:.2f}% | Robust: {std_rob_acc:.2f}%")

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 56.55it/s]


Results Summary
--------------------------------------------------
IBP Model     - Standard: 95.24% | Robust: 83.81%
Standard Model- Standard: 97.22% | Robust: 1.03%


### Part B

In [ ]:
def verify_robustness_single_image(model, x, y, eps):
    x_flat = x.view(1, -1)
    lower_bound = torch.clamp(x_flat - eps, 0.0, 1.0)
    upper_bound = torch.clamp(x_flat + eps, 0.0, 1.0)
    input_bounds = HyperRectangle(lower_bound, upper_bound)

    # Build sequential model for IBP propagation
    layers = [
        model.fc1, nn.ReLU(),
        model.fc2, nn.ReLU(),
        model.fc3, nn.ReLU(),
        model.fc4
    ]
    bound_model = nn.Sequential(*layers)

    # Construct bounded model and compute output bounds
    bounded_net = BoundModelFactory().build(bound_model)
    bounds = bounded_net.ibp(input_bounds)

    lower, upper = bounds.lower[0], bounds.upper[0]
    true_lower = lower[y]

    # Verify that all other classes’ upper bounds stay below the true label’s lower bound
    return not any(upper[i] >= true_lower for i in range(len(lower)) if i != y)

In [ ]:
def compute_verified_accuracy(model, loader, eps, device="cuda"):
    model.eval()
    verified = correct = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            preds = model(x).argmax(dim=1)

        for i in range(len(x)):
            if preds[i].item() == y[i].item():
                correct += 1
                if verify_robustness_single_image(model, x[i:i+1], y[i].item(), eps):
                    verified += 1

    acc = 100.0 * verified / correct if correct > 0 else 0.0
    return verified, correct, acc

In [ ]:
epsilon_values = np.linspace(0.01, 0.1, 10)
verified_results = []

print("Verified Accuracy Evaluation (Box Verification)")

for eps in epsilon_values:
    num_verified, num_correct, verified_acc = compute_verified_accuracy(model, test_loader, eps, device=device)
    verified_results.append((eps, num_verified, num_correct, verified_acc))
    print(f"ε={eps:.3f} → Verified: {num_verified}/{num_correct} ({verified_acc:.2f}%)")

Verified Accuracy Evaluation (Box Verification)
ε=0.010 → Verified: 9519/9602 (99.14%)
ε=0.020 → Verified: 9419/9602 (98.09%)
ε=0.030 → Verified: 9323/9602 (97.09%)
ε=0.040 → Verified: 9196/9602 (95.77%)
ε=0.050 → Verified: 9041/9602 (94.16%)
ε=0.060 → Verified: 8877/9602 (92.45%)
ε=0.070 → Verified: 8664/9602 (90.23%)
ε=0.080 → Verified: 8419/9602 (87.68%)
ε=0.090 → Verified: 8164/9602 (85.02%)
ε=0.100 → Verified: 7764/9602 (80.86%)
